# 96 — GNN Edge Classifier Test

Heterogeneous GNN (`HeteroConv` + `NNConv`) for directed attack-edge classification.
2-planet scenario: one owned by player 0, one neutral.
Label = 1 if my ships > neutral ships at step 0.

In [ ]:
%run 96-library.py
import torch
import torch.nn as nn
from torch_geometric.nn import SAGEConv, NNConv
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import copy, math, random

In [ ]:
_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}

def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')
    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100); ax.set_ylim(100, 0)
        ax.set_aspect('equal'); ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values(): sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, '+'+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, ang, from_id, ships = f
            ax.plot(x, y, 'D', color=_COLORS.get(owner,'#888888'), markersize=5, zorder=5)
        return []
    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

In [ ]:
data0, snaps0 = generate_sample(42)
print("master x:", data0['master'].x.shape)
print("planet x:", data0['planet'].x.shape)
print("attack edge_index:", data0['planet','attacks','planet'].edge_index.shape)
print("attack edge_attr: ", data0['planet','attacks','planet'].edge_attr.shape)
print("label:", int(data0.y.item()))
make_animation(snaps0, title='Sample seed=42', interval=200)

In [ ]:
print("Generating train dataset (100 samples)...")
train_dataset = [generate_sample(i)      for i in range(100)]
print("Generating test dataset (10 samples)...")
test_dataset  = [generate_sample(1000+i) for i in range(10)]
train_labels = [int(d.y.item()) for d, _ in train_dataset]
test_labels  = [int(d.y.item()) for d, _ in test_dataset]
print(f"Train label distribution: {train_labels.count(1)} pos / {train_labels.count(0)} neg")
print(f"Test  label distribution: {test_labels.count(1)} pos / {test_labels.count(0)} neg")